In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_3.py ──
"""
Shared infrastructure for MLFP04 Exercise 3 — Dimensionality Reduction.

Contains: data loading, scaling, common output directory, KMeans-based
silhouette evaluation in the embedding space. Technique-specific code
(PCA/KPCA/t-SNE/UMAP algorithms and their plots) lives in the per-
technique files, NOT here.

"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT + REPRODUCIBILITY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex3_dimreduce"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEFAULT_N_CLUSTERS = 4

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customers (reused from MLFP03)
# ════════════════════════════════════════════════════════════════════════


def load_customer_matrix() -> tuple[np.ndarray, list[str], pl.DataFrame]:
    """Load e-commerce customers, standardise numeric features.

    Returns:
        X          : (n_samples, n_features) standardised float matrix
        feature_cols: list of feature column names in order
        df_raw     : the raw polars DataFrame before scaling
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
        and c not in ("customer_id",)
    ]

    df_clean = customers.drop_nulls(subset=feature_cols)
    X_raw, _, _ = to_sklearn_input(df_clean, feature_columns=feature_cols)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    return X, feature_cols, df_clean


# ════════════════════════════════════════════════════════════════════════
# EMBEDDING-SPACE CLUSTER QUALITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_embedding_silhouette(
    embedding: np.ndarray,
    n_clusters: int = DEFAULT_N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> float:
    """Fit KMeans in the embedding space and return the silhouette score.

    This is the standard "does the reducer preserve structure?" probe used
    across all five technique files. Returns -1.0 when only one cluster is
    found (e.g. collapsed embedding).
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km.fit_predict(embedding)
    if len(set(labels)) < 2:
        return -1.0
    return float(silhouette_score(embedding, labels))


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — used by KPCA / t-SNE / UMAP / Isomap for kernel-cost paths
# ════════════════════════════════════════════════════════════════════════


def subsample_indices(
    n_samples: int, n_target: int, random_state: int = RANDOM_STATE
) -> np.ndarray:
    """Deterministic subsample indices for expensive O(n^2) methods."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, min(n_target, n_samples), replace=False)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 3.5: Method comparison + intrinsic dimensionality
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compare PCA, Kernel PCA, t-SNE, UMAP, Isomap on one ruler
#   - Estimate intrinsic dimensionality (variance, Kaiser, NN MLE)
#   - Pick the right reducer given production vs visualisation goals
#
# PREREQUISITES: 01-04_*.py (all four previous files).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — why intrinsic dim is the right target
#   2. Build — five reducers on the same data
#   3. Train — compute silhouette for every configuration
#   4. Visualise — leaderboard + intrinsic-dimensionality summary
#   5. Apply — GovTech Singapore FormSG respondent segmentation
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE, Isomap
from sklearn.neighbors import NearestNeighbors

from kailash_ml import ModelVisualizer


try:
    import umap as umap_lib  # type: ignore

    UMAP_AVAILABLE = True
except ImportError:  # pragma: no cover
    umap_lib = None
    UMAP_AVAILABLE = False



## THEORY — intrinsic dimensionality


In [ ]:
# Your data has p ambient dimensions but varies along d << p independent
# axes. If d is small, dim-reduction is nearly free. If d ≈ p, it hurts.
# Estimate d BEFORE picking a method.



## TASK 2 — BUILD: shared preprocessing + pre-reduction


In [ ]:
X, feature_cols, _ = load_customer_matrix()
n_samples, n_features = X.shape
print(f"=== E-commerce customers ===  n={n_samples:,}, p={n_features}")

# TODO: fit a full-rank PCA on X. We need it for intrinsic-dim estimates
# AND for the pre-reduced inputs to t-SNE/UMAP.
pca_full = ____
pca_full.fit(X)
evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)
explained_variance = pca_full.explained_variance_

n_80 = int(np.searchsorted(cum_evr, 0.80) + 1)
n_90 = int(np.searchsorted(cum_evr, 0.90) + 1)
n_95 = int(np.searchsorted(cum_evr, 0.95) + 1)

X_pca10 = pca_full.transform(X)[:, : min(10, n_features)]
idx = subsample_indices(n_samples, n_target=3000)



## TASK 3 — TRAIN: silhouette across every reducer


In [ ]:
method_silhouettes: dict[str, float] = {}

# (a) PCA at 2D, 80%, 90%, 95% variance
for n_comp in [2, n_80, n_90, n_95]:
    pca_test = PCA(n_components=n_comp, random_state=42)
    X_test = pca_test.fit_transform(X)
    method_silhouettes[f"PCA {n_comp}d"] = evaluate_embedding_silhouette(X_test)

# (b) Kernel PCA — three configs on the subsample
for kernel, params, label in [
    ("rbf", {"gamma": 0.1}, "KernelPCA rbf g=0.1"),
    ("rbf", {"gamma": 1.0}, "KernelPCA rbf g=1.0"),
    ("poly", {"degree": 3, "gamma": 0.1}, "KernelPCA poly d=3"),
]:
    # TODO: build and fit a KernelPCA(n_components=8, kernel=..., **params)
    # on X[idx], then record its silhouette.
    kpca = ____
    X_kpca = ____
    method_silhouettes[label] = evaluate_embedding_silhouette(X_kpca)

# (c) t-SNE at a few perplexities
for perp in [15, 30, 50]:
    # TODO: fit TSNE(n_components=2, perplexity=perp, max_iter=1000,
    # random_state=42, init='pca', learning_rate='auto') on X_pca10[idx].
    tsne = ____
    emb = ____
    method_silhouettes[f"t-SNE p={perp}"] = evaluate_embedding_silhouette(emb)

# (d) UMAP — three configs
if UMAP_AVAILABLE:
    for n_nbr, min_d, label in [
        (15, 0.1, "UMAP default"),
        (50, 0.5, "UMAP global"),
        (15, 0.0, "UMAP tight"),
    ]:
        reducer = umap_lib.UMAP(
            n_components=2,
            n_neighbors=n_nbr,
            min_dist=min_d,
            random_state=42,
            metric="euclidean",
        )
        reducer.fit(X_pca10[idx])
        emb_full = reducer.transform(X_pca10)
        method_silhouettes[label] = evaluate_embedding_silhouette(emb_full)

# (e) Isomap — manifold learning reference
iso = Isomap(n_components=2, n_neighbors=10)
X_iso = iso.fit_transform(X_pca10[idx])
method_silhouettes["Isomap k=10"] = evaluate_embedding_silhouette(X_iso)



### Checkpoint 1


In [ ]:
assert (
    len(method_silhouettes) >= 10
), f"Expected >=10 method configurations, got {len(method_silhouettes)}"
print(
    f"\n[ok] Checkpoint 1 — {len(method_silhouettes)} reducer configurations scored\n"
)



## TASK 4 — VISUALISE: leaderboard + intrinsic dimensionality


Levina-Bickel MLE estimator of intrinsic dimension.


In [ ]:
print("=== Leaderboard (descending silhouette) ===")
for name, sil in sorted(method_silhouettes.items(), key=lambda x: -x[1]):
    print(f"  {name:<30}: {sil:+.4f}")

viz = ModelVisualizer()
fig = viz.metric_comparison(
    {name: {"Silhouette": sil} for name, sil in method_silhouettes.items()}
)
fig.update_layout(title="Dimensionality reduction: cluster-quality leaderboard")
leaderboard_path = OUTPUT_DIR / "05_leaderboard.html"
fig.write_html(str(leaderboard_path))
print(f"\nSaved: {leaderboard_path}")


# Intrinsic dimensionality estimators
# TODO: Kaiser — count eigenvalues greater than 1.
n_kaiser = ____

broken_stick = np.array(
    [
        sum(1.0 / j for j in range(i, n_features + 1)) / n_features
        for i in range(1, n_features + 1)
    ]
)
n_broken = int((evr > broken_stick).sum())


def estimate_intrinsic_dim_nn(
    X: np.ndarray, k_values: list[int], n_sub: int = 1000
) -> float:
    rng = np.random.default_rng(42)
    sub = rng.choice(len(X), min(n_sub, len(X)), replace=False)
    X_s = X[sub]
    nn = NearestNeighbors(n_neighbors=max(k_values)).fit(X_s)
    dists, _ = nn.kneighbors(X_s)
    log_ratios = []
    for k in k_values:
        d_k = dists[:, k - 1]
        d_1 = dists[:, 0]
        valid = (d_k > 0) & (d_1 > 0)
        if valid.sum() > 10:
            log_ratios.append(float(np.mean(np.log(d_k[valid] / d_1[valid]))))
    if not log_ratios:
        return float("nan")
    m = float(np.mean(log_ratios))
    return 1.0 / m if m > 0 else float("nan")


intrinsic_mle = estimate_intrinsic_dim_nn(X, k_values=[5, 10, 20, 30])

print("\n=== Intrinsic dimensionality estimates ===")
print(f"  Ambient (p)          : {n_features}")
print(f"  PCA 80% variance     : {n_80}")
print(f"  PCA 90% variance     : {n_90}")
print(f"  PCA 95% variance     : {n_95}")
print(f"  Kaiser (eig > 1)     : {n_kaiser}")
print(f"  Broken-stick         : {n_broken}")
print(f"  NN MLE (Levina-Bickel): {intrinsic_mle:.1f}")



### Checkpoint 2


In [ ]:
assert n_90 <= n_features, "intrinsic dim cannot exceed ambient"
assert 1 <= n_kaiser <= n_features
print("\n[ok] Checkpoint 2 — intrinsic dimensionality estimated via 4 methods\n")



## TASK 5 — APPLY: GovTech Singapore FormSG respondent segmentation


In [ ]:
# SCENARIO: GovTech's FormSG serves ~5M form submissions/yr. Three
# audiences need different reducers from the same data:
#   - PM deck         -> t-SNE picture (micro-segments)
#   - DS team         -> PCA features (invertible, stable)
#   - Production ML   -> UMAP embedder (out-of-sample)
# Segmentation-driven UX fixes cut form abandonment by 14% -> ~700K more
# completed forms/yr -> ~S$8.4M/yr in avoided manual handling.
# WORKFLOW: run this notebook on the monthly snapshot, use intrinsic dim
# to bound expectations, pick per audience from the leaderboard.

top_method, top_sil = max(method_silhouettes.items(), key=lambda kv: kv[1])
print(f"\n=== GovTech FormSG projection ===")
print(f"  Top reducer on this dataset : {top_method}  (silhouette {top_sil:+.4f})")
print(f"  Intrinsic dim (NN MLE)      : {intrinsic_mle:.1f}")
print(f"  Ambient dim                 : {n_features}")



## DECISION GUIDE


PRODUCTION:   PCA first, UMAP if PCA is insufficient.
  VISUALISE:    t-SNE for dense micro-clusters, UMAP for mixed scales.
  EXPLAIN:      PCA — it's the only one with true inverse_transform.


In [ ]:
print(
)



## REFLECTION


[x] Ran five reducer families on the same dataset with one metric
  [x] Built a silhouette leaderboard across all configurations
  [x] Estimated intrinsic dimensionality with four methods
  [x] Picked per-audience reducers for a real GovTech use case

  KEY INSIGHT: There is no "best" dimensionality reducer — only a best
  reducer FOR A SPECIFIC AUDIENCE AND DOWNSTREAM TASK.

  Exercise 3 complete. Next: Exercise 4 — anomaly detection ensembles.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

